In [ ]:
import pandas as pd
import numpy as np
from feature_analysis import find_high_correlation_pairs

df = pd.read_csv('brfss_2024.zip', compression='zip')
pd.set_option('display.max_columns', None)

target_col = 'CVDSTRK3'

# Keep only rows where we have a clear stroke status
df = df[(df[target_col] == 1) | (df[target_col] == 2)]

print("Amount of rows and features:", df.shape)
df.head(1)

In [ ]:
# Manual feature selection - the goal is to drop features such as administrative and survey design variables.
# These variables tell how the data was collected, not what respondents reported about their health or behaviour.

df.drop(columns=[
    # When the survey was conducted and general identifiers whether is eligible to be included in the survey
    "FMONTH", "IDATE", "IMONTH", "IDAY", "IYEAR", "DISPCODE", "SEQNO", "_PSU", "SAFETIME",
    "LADULT1", "CADULT1", "RESPSLC1", "CTELENM1", "CTELNUM1","CELPHON1", "CELLFON5", "LANDLINE",
    "LANDSEX3", "CELLSEX3",
    
    # Residency and household variables
    "PVTRESD1", "PVTRESD3", "COLGHOUS", "STATERE1", "CSTATE1", "NUMADULT", 
    "HHADULT", "NUMHHOL4", "NUMPHON4", "CPDEMO1C", "CCLGHOUS", "GUNLOAD", 
    "LOADULK2", "FIREARM5", "_CHLDCNT", "CHILDREN", "CAGEG", "RENTHOM1",

    # Survey language and version variables
    "QSTVER", "QSTLANG", "ICFQSTVR",

    # Statistical sampling weights & design variables
    "_STSTR", "_STRWT", "_RAWRAKE", "_WT2RAKE", "_CLLCPWT", 
    "_DUALUSE", "_DUALCOR", "_LLCPWT2", "_LLCPWT",

    # Post Course of Treatment questions
    "CSRVINST", "CSRVINSR", "CSRVDEIN", "PCSTALK2", "CSRVSUM",

    # Caregiver - about the respondent's relationship to their caregiver, 
    # not about the respondent's health or behaviour.
    "CAREGIV1", "CRGVREL5", "CRGVPRB4", "CRGVALZD", 
    "CRGVNURS", "CRGVPER2", "CRGVHOU2", "CRGVHRS2", "CRGVLNG2", 

    # Family Planning
    "RCSRLTN2", "HADSEX", "PFPPRVN4", "TYPCNTR9", "NOBCUSE8",

    "RCSXBRTH", "RCSGEND1", "HPVDSHT", "CSRVCTL2",
], inplace=True)

print("Dataset after dropping columns that are unrelated to health outcomes by design:", df.shape)
df.head(1)

In [ ]:
# Find highly correlated features using Cramér's V for categorical variables.
# from feature_analysis import find_high_correlation_pairs

# pairs_df = find_high_correlation_pairs(df, threshold=0.85, sample_n = 50_000, random_state = 42)
# print(pairs_df.to_string(index=False))

In [ ]:
# Remove redundant features

df.drop(columns=[
    # General health / days unhealthy
    # Keep: GENHLTH, PHYSHLTH, MENTHLTH, _TOTINDA 
    # Precise amount of days unhealthy in the past 30 days, while the alternatives are coarser categories or binary variables
    "_RFHLTH", "_PHYS14D", "_MENT14D", "POORHLTH", "EXERANY2",

    # BMI / anthropometrics
    # Keep: _BMI5 (continuous computed BMI)
    # Raw height and weight in all unit encodings are redundant as we have _BMI5. 
    # _BMI5CAT and _RFBMI5 are redundant to _BMI5.
    "WEIGHT2", "HEIGHT3", "HTIN4", "HTM4", 
    "WTKG3", "_BMI5CAT", "_RFBMI5",

    # Race / ethnicity
    # Keep: _RACE (8-category combined variable covering all race variants)
    "_RACEG21", "_RACEGR3", "_RACEPRV", "_MRACE1", 
    "_IMPRACE", "_CHISPNC", "_CRACE1", "_HISPANC",

    # Age
    # Keep: _AGE80 (Alternatives are encoded in age groups, but _AGE80 is a continuous number 
    # that gives us more precise information about the respondent's age)
    "_AGE65YR", "_AGE_G", "_AGEG5YR", "_LCSAGE",

    # Smoking
    # Keep: _SMOKER3 (Derived variable that combines current / former / never / every day smoking status)
    # Keep: LCSNUMC_ (preciese number of cigarettes smoked per day)
    # Raw inputs SMOKE100 and SMOKDAY2 are fully absorbed by _SMOKER3.
    # _RFSMOK3 is a binary collapse of _SMOKER3. USENOW3, LASTSMK2,
    "_RFSMOK3", "USENOW3", "LASTSMK2", "STOPSMK2", "LCSNUMCG",
    "HEATTBCO", "SMOKE100", "SMOKDAY2", "_PACKDAY", "_LCSSMKG",
    "_PACKYRS",

    # E-cigarettes / vaping
    # Keep: ECIGNOW3 (Has all 3-category current / former / never status)
    "_CURECI3", "MENTCIGS", "MENTECIG",
    
    # Marijuana / cannabis
    # Keep: MARIJAN1 (Give us not just the status but precise amount of consumption days)
    "MARJEAT", "MARJDAB", "MARJOTHR", "MARJSMOK", "MARJVAPE", "USEMRJN4",

    # Alcohol
    # Keep: _DRNKWK3 (Calculated total number of alcoholic beverages consumed per week, 
    # while the alternatives are coarser categories or binary variables)
    "_RFBING6", "DRNKANY6", "ALCDAY4", "DRNK3GE5", 
    "MAXDRNKS", "DROCDY4_", "_RFDRHV9", "AVEDRNK4",

    # Education / Income
    # Keep: EDUCA / INCOME3 (EDUCA and INCOME3 respectively provide a more detailed range of education and income levels)
    "_EDUCAG", "_INCOMG1", 

    # Health insurance / access
    # Keep: PRIMINS2 , PERSDOC3, CHECKUP1
    "MEDCOST1", "_HLTHPL2" , "_HCVU654",

    # Sex
    # Keep: _SEX
    "SEXVAR",

    # Asthma
    # Keep: _ASTHMS1 (Computed Asthma Status, covers all 3 current, former and never asthma status)
    "_LTASTH1", "_CASTHM1", "CASTHDX2", "CASTHNO2", "ASTHNOW", "ASTHMA3", 

    # Cancer / Lung cancer screening test
    # Keep: _LCSCTSN, CERVSCRN, 
    "CSRVDOC1", "CSRVRTRN", "CSRVCLIN", "LCSSCNCR", "LCSCTSC1", "_LCSPSTF",
    "LCSCTWHN", 

    # Dental / oral health
    # Keep: LASTDEN4, RMVTETH4
    "_EXTETH3", "_DENVST3", "_ALTETH3",

    # PSA test follow-ups
    # Keep: PSATEST1 (ever had PSA test)
    "PSATIME1", "PCPSARS2", "PSASUGS1",

    # Mammography / cervical screening
    # Keep: HOWLONG
    "_SGMS102", "_SGMSCP2", "_CLNSCP2", "_MAM402Y", "_RFMAM23",
    "_CRVSCRN", "_RFPAP37",

    # Flu shot
    # FLUSHOT7 (as it has no age restriction, while _FLSHOT7 is only for 65+)
    "_FLSHOT7", "_PNEUMO3", "FLSHTMY3", "IMFVPLA5", "HIVTSTD3",

    # Colorectal screening
    # Keep: COLNSIGM (handles both sigmoidoscopy and colonoscopy, as well as both)
    "_HADSIGM", "_HADCOLN", "VIRCOLO1", "_VIRCOL2", "_RFBLDS6", "_STOLDN2",
    "STOOLDN2", "_CRCREC3", "BLDSTFIT", "SDNATES1", "VCLNTES2", "SMALSTOL",
    "_SBONTI2", "HADSIGM4", "LASTSIG4",
    
    # Arthritis
    # Keep: HAVARTH4
    "_DRDXAR2", "ARTHEXER", "CHKHEMO3", 

    # Urbanicity
    # Keep: _URBSTAT
    "_METSTAT", "MSCODE",

    # HIV/AIDS
    # Keep: HIVTST7
    "_AIDTST4", "_HPV5YR1", "_PAPHPV1", "HPVADSH1",

    # Angina or Coronary Heart Disease
    # Keep: _MICHD and CVDINFR4
    "CVDCRHD4",
], inplace=True)

print("Dataset after dropping redundant features:", df.shape)
df.head(1)

In [ ]:
# Compare the results of correlation analysis after removing redundant features

pairs_df = find_high_correlation_pairs(df, threshold=0.80, sample_n = 50_000, random_state = 42)
print(pairs_df.to_string(index=False))

In [ ]:
# The most common values:
# 7,77,777 - don`t know / unsure / refused to answer
# 8,88,888 - none / never

df["GENHLTH"] = df["GENHLTH"].replace({7: np.nan, 9: np.nan})
df["PHYSHLTH"] = df["PHYSHLTH"].replace({77: np.nan, 88: 0, 99: np.nan})
df["MENTHLTH"] = df["MENTHLTH"].replace({77: np.nan, 88: 0, 99: np.nan})
df["PRIMINS2"] = df["PRIMINS2"].replace({77: np.nan, 88: 0, 99: np.nan})
df["PERSDOC3"] = df["PERSDOC3"].replace({7: np.nan, 9: np.nan})
df["CHECKUP1"] = df["CHECKUP1"].replace({7: np.nan, 8: 0, 9: np.nan})
df["LASTDEN4"] = df["LASTDEN4"].replace({7: np.nan, 8: 0, 9: np.nan})
df["RMVTETH4"] = df["RMVTETH4"].replace({7: np.nan, 8: 0, 9: np.nan})
df["CVDINFR4"] = df["CVDINFR4"].replace({7: np.nan, 9: np.nan})
df["CHCSCNC1"] = df["CHCSCNC1"].replace({7: np.nan, 9: np.nan})
df["CHCOCNC1"] = df["CHCOCNC1"].replace({7: np.nan, 9: np.nan})
df["CHCCOPD3"] = df["CHCCOPD3"].replace({7: np.nan, 9: np.nan})
df["ADDEPEV3"] = df["ADDEPEV3"].replace({7: np.nan, 9: np.nan})
df["CHCKDNY2"] = df["CHCKDNY2"].replace({7: np.nan, 9: np.nan})
df["HAVARTH4"] = df["HAVARTH4"].replace({7: np.nan, 9: np.nan})
df["DIABETE4"] = df["DIABETE4"].replace({7: np.nan, 9: np.nan})
df["DIABAGE4"] = df["DIABAGE4"].replace({98: np.nan, 99: np.nan})
df["MARITAL"] = df["MARITAL"].replace({9: np.nan})
df["EDUCA"] = df["EDUCA"].replace({9: np.nan})
df["VETERAN3"] = df["VETERAN3"].replace({7: np.nan, 9: np.nan})
df["EMPLOY1"] = df["EMPLOY1"].replace({9: np.nan})
df["INCOME3"] = df["INCOME3"].replace({77: np.nan, 99: np.nan})
df["PREGNANT"] = df["PREGNANT"].replace({7: np.nan, 9: np.nan})
df["DEAF"] = df["DEAF"].replace({7: np.nan, 9: np.nan})
df["BLIND"] = df["BLIND"].replace({7: np.nan, 9: np.nan})
df["DECIDE"] = df["DECIDE"].replace({7: np.nan, 9: np.nan})
df["DIFFWALK"] = df["DIFFWALK"].replace({7: np.nan, 9: np.nan})
df["DIFFWALK"] = df["DIFFWALK"].replace({7: np.nan, 9: np.nan})
df["DIFFDRES"] = df["DIFFDRES"].replace({7: np.nan, 9: np.nan})
df["DIFFALON"] = df["DIFFALON"].replace({7: np.nan, 9: np.nan})
df["HADMAM"] = df["HADMAM"].replace({7: np.nan, 9: np.nan})
df["HOWLONG"] = df["HOWLONG"].replace({7: np.nan, 9: np.nan})
df["CERVSCRN"] = df["CERVSCRN"].replace({7: np.nan, 9: np.nan})
df["CRVCLCNC"] = df["CRVCLCNC"].replace({7: np.nan, 9: np.nan})
df["CRVCLPAP"] = df["CRVCLPAP"].replace({7: np.nan, 9: np.nan})
df["CRVCLHPV"] = df["CRVCLHPV"].replace({7: np.nan, 9: np.nan})
df["HADHYST2"] = df["HADHYST2"].replace({7: np.nan, 9: np.nan})
df["COLNSIGM"] = df["COLNSIGM"].replace({7: np.nan, 9: np.nan})
df["COLNTES1"] = df["COLNTES1"].replace({7: np.nan, 9: np.nan})
df["SIGMTES1"] = df["SIGMTES1"].replace({7: np.nan, 9: np.nan})
df["COLNCNCR"] = df["COLNCNCR"].replace({7: np.nan, 9: np.nan})
df["STOLTEST"] = df["STOLTEST"].replace({7: np.nan, 9: np.nan})
df["ECIGNOW3"] = df["ECIGNOW3"].replace({7: np.nan, 9: np.nan})
df["LCSFIRST"] = df["LCSFIRST"].replace({777: np.nan, 888: -1, 999: np.nan})
df["FLUSHOT7"] = df["FLUSHOT7"].replace({7: np.nan, 9: np.nan})
df["PNEUVAC4"] = df["PNEUVAC4"].replace({7: np.nan, 9: np.nan})
df["HIVTST7"] = df["HIVTST7"].replace({7: np.nan, 9: np.nan})
df["HIVRISK5"] = df["HIVRISK5"].replace({7: np.nan, 9: np.nan})
df["PDIABTS1"] = df["PDIABTS1"].replace({7: np.nan, 8: 0, 9: np.nan})
df["PREDIAB2"] = df["PREDIAB2"].replace({7: np.nan, 9: np.nan})
df["DIABTYPE"] = df["DIABTYPE"].replace({7: np.nan, 9: np.nan})
df["INSULIN1"] = df["INSULIN1"].replace({7: np.nan, 9: np.nan})
df["EYEEXAM1"] = df["EYEEXAM1"].replace({6: np.nan, 7: np.nan, 8: 0, 9: np.nan})
df["DIABEYE1"] = df["DIABEYE1"].replace({7: np.nan, 8: 0, 9: np.nan})
df["DIABEDU1"] = df["DIABEDU1"].replace({7: np.nan, 8: 0, 9: np.nan})
df["FEETSORE"] = df["FEETSORE"].replace({7: np.nan, 9: np.nan})
df["SHINGLE2"] = df["SHINGLE2"].replace({7: np.nan, 9: np.nan})
df["HPVADVC4"] = df["HPVADVC4"].replace({7: np.nan, 9: np.nan})
df["TETANUS1"] = df["TETANUS1"].replace({7: np.nan, 9: np.nan})
df["CNCRDIFF"] = df["CNCRDIFF"].replace({7: np.nan, 9: np.nan})
df["CNCRAGE"] = df["CNCRAGE"].replace({98: np.nan, 99: np.nan})
df["CNCRTYP2"] = df["CNCRTYP2"].replace({77: np.nan, 99: np.nan})
df["CSRVTRT3"] = df["CSRVTRT3"].replace({7: np.nan, 9: np.nan})
df["CSRVPAIN"] = df["CSRVPAIN"].replace({7: np.nan, 9: np.nan})
df["PSATEST1"] = df["PSATEST1"].replace({7: np.nan, 9: np.nan})
df["CIMEMLO1"] = df["CIMEMLO1"].replace({7: np.nan, 9: np.nan})
df["CDWORRY"] = df["CDWORRY"].replace({7: np.nan, 9: np.nan})
df["CDDISCU1"] = df["CDDISCU1"].replace({7: np.nan, 9: np.nan})
df["CDHOUS1"] = df["CDHOUS1"].replace({7: np.nan, 9: np.nan})
df["CDSOCIA1"] = df["CDSOCIA1"].replace({7: np.nan, 9: np.nan})
df["ACEDEPRS"] = df["ACEDEPRS"].replace({7: np.nan, 9: np.nan})
df["ACEDRINK"] = df["ACEDRINK"].replace({7: np.nan, 9: np.nan})
df["ACEDRUGS"] = df["ACEDRUGS"].replace({7: np.nan, 9: np.nan})
df["ACEPRISN"] = df["ACEPRISN"].replace({7: np.nan, 9: np.nan})
df["ACEDIVRC"] = df["ACEDIVRC"].replace({7: np.nan, 8: 3, 9: np.nan})
df["ACEPUNCH"] = df["ACEPUNCH"].replace({7: np.nan, 9: np.nan})
df["ACEHURT1"] = df["ACEHURT1"].replace({7: np.nan, 9: np.nan})
df["ACESWEAR"] = df["ACESWEAR"].replace({7: np.nan, 9: np.nan})
df["ACETOUCH"] = df["ACETOUCH"].replace({7: np.nan, 9: np.nan})
df["ACETTHEM"] = df["ACETTHEM"].replace({7: np.nan, 9: np.nan})
df["ACEHVSEX"] = df["ACEHVSEX"].replace({7: np.nan, 9: np.nan})
df["ACEADSAF"] = df["ACEADSAF"].replace({7: np.nan, 9: np.nan})
df["ACEADNED"] = df["ACEADNED"].replace({7: np.nan, 9: np.nan})
df["LSATISFY"] = df["LSATISFY"].replace({7: np.nan, 9: np.nan})
df["EMTSUPRT"] = df["EMTSUPRT"].replace({7: np.nan, 9: np.nan})
df["SDLONELY"] = df["SDLONELY"].replace({7: np.nan, 9: np.nan})
df["SDHEMPLY"] = df["SDHEMPLY"].replace({7: np.nan, 9: np.nan})
df["FOODSTMP"] = df["FOODSTMP"].replace({7: np.nan, 9: np.nan})
df["SDHFOOD1"] = df["SDHFOOD1"].replace({7: np.nan, 9: np.nan})
df["SDHBILLS"] = df["SDHBILLS"].replace({7: np.nan, 9: np.nan})
df["SDHUTILS"] = df["SDHUTILS"].replace({7: np.nan, 9: np.nan})
df["SDHTRNSP"] = df["SDHTRNSP"].replace({7: np.nan, 9: np.nan})
df["HOWSAFE1"] = df["HOWSAFE1"].replace({7: np.nan, 9: np.nan})
df["MARIJAN1"] = df["MARIJAN1"].replace({77: np.nan, 88: 0, 99: np.nan})
df["SSBSUGR2"] = df["SSBSUGR2"].replace({777: np.nan, 888: 0, 999: np.nan})
df["SSBFRUT3"] = df["SSBFRUT3"].replace({777: np.nan, 888: 0, 999: np.nan})
df["SOMALE"] = df["SOMALE"].replace({7: np.nan, 9: np.nan})
df["SOFEMALE"] = df["SOFEMALE"].replace({7: np.nan, 9: np.nan})
df["_TOTINDA"] = df["_TOTINDA"].replace({9: np.nan})
df["_ASTHMS1"] = df["_ASTHMS1"].replace({9: np.nan})
df["_RACE"] = df["_RACE"].replace({9: np.nan})
df["_SMOKER3"] = df["_SMOKER3"].replace({9: np.nan})
df["_DRNKWK3"] = df["_DRNKWK3"].replace({99900: np.nan})

In [ ]:
# Handle dont't know / unsure / refused / none / never

# 7/9 → NaN (don't know / refused)
_cols_dk = [
    "GENHLTH", "PERSDOC3", "CVDINFR4", "CHCSCNC1", "CHCOCNC1", "CHCCOPD3",
    "ADDEPEV3", "CHCKDNY2", "HAVARTH4", "DIABETE4", "MARITAL", "VETERAN3",
    "PREGNANT", "DEAF", "BLIND", "DECIDE", "DIFFWALK", "DIFFDRES", "DIFFALON",
    "HADMAM", "HOWLONG", "CERVSCRN", "CRVCLCNC", "CRVCLPAP", "CRVCLHPV",
    "HADHYST2", "COLNSIGM", "COLNTES1", "SIGMTES1", "COLNCNCR", "STOLTEST",
    "ECIGNOW3", "FLUSHOT7", "PNEUVAC4", "HIVTST7", "HIVRISK5", "PREDIAB2",
    "DIABTYPE", "INSULIN1", "FEETSORE", "SHINGLE2", "HPVADVC4", "TETANUS1",
    "CNCRDIFF", "CSRVTRT3", "CSRVPAIN", "PSATEST1", "CIMEMLO1", "CDWORRY",
    "CDDISCU1", "CDHOUS1", "CDSOCIA1", "ACEDEPRS", "ACEDRINK", "ACEDRUGS",
    "ACEPRISN", "ACEPUNCH", "ACEHURT1", "ACESWEAR", "ACETOUCH", "ACETTHEM",
    "ACEHVSEX", "ACEADSAF", "ACEADNED", "LSATISFY", "EMTSUPRT", "SDLONELY",
    "SDHEMPLY", "FOODSTMP", "SDHFOOD1", "SDHBILLS", "SDHUTILS", "SDHTRNSP",
    "HOWSAFE1", "SOMALE", "SOFEMALE"
]

for col in _cols_dk:
    df[col] = df[col].replace({7: np.nan, 9: np.nan})

# 7/9 → NaN, 8 → 0 (none/never)
_cols_dk_none = [
    "CHECKUP1", "LASTDEN4", "RMVTETH4", "DIABEYE1", "DIABEDU1", "PDIABTS1",
]
for col in _cols_dk_none:
    df[col] = df[col].replace({7: np.nan, 8: 0, 9: np.nan})

# 77/99 → NaN (no "none" code)
for col in ["PHYSHLTH", "MENTHLTH"]:
    df[col] = df[col].replace({77: np.nan, 99: np.nan})

# 77/88/99 → NaN/0/NaN
for col in ["PRIMINS2"]:
    df[col] = df[col].replace({77: np.nan, 88: 0, 99: np.nan})

# 77/99 → NaN (no 88 code)
for col in ["CNCRTYP2", "INCOME3"]:
     df[col] = df[col].replace({77: np.nan, 99: np.nan})

# 777/888/999 → NaN/0/NaN
for col in ["SSBSUGR2", "SSBFRUT3", "MARIJAN1"]:
    df[col] = df[col].replace({777: np.nan, 888: 0, 999: np.nan})

# 98/99 → NaN (age fields)
for col in ["DIABAGE4", "CNCRAGE"]:
    df[col] = df[col].replace({98: np.nan, 99: np.nan})

# 9 → NaN (where 9 stands for Don`t know/Refused/Missing)
for col in ["_TOTINDA", "_ASTHMS1", "_RACE", "_SMOKER3", "EDUCA"]:
    df[col] = df[col].replace({9: np.nan})

# Irregular codings — handled individually
df["ACEDIVRC"] = df["ACEDIVRC"].replace({7: np.nan, 8: 3, 9: np.nan})
df["EYEEXAM1"] = df["EYEEXAM1"].replace({6: np.nan, 7: np.nan, 8: 0, 9: np.nan})
df["LCSFIRST"] = df["LCSFIRST"].replace({777: np.nan, 888: -1, 999: np.nan})
df["_DRNKWK3"] = df["_DRNKWK3"].replace({99900: np.nan})